# Exploration

This notebook contains code snippets and experimental steps. Some code may not run as intended because the workflow has been updated and finalized in other notebooks.

## Regression Decision Tree

## Model Attempts Overview

I ran a series of experiments to test the model’s performance under different feature sets and configurations:

- **Attempt 1:** Use all features, including Flesch-Kincaid (high accuracy, but overreliant on one metric).
- **Attempt 2:** Remove Flesch-Kincaid score to encourage use of other features.
- **Attempt 3:** Add engineered features for a broader, more robust predictor.

For each attempt, I show the feature selection and report key metrics.

## Attempt 1: All Features (Including Flesch-Kincaid)

The model uses all available features, including the Flesch-Kincaid score. This results in high performance, but the model is dominated by a single metric, reducing generalisability.

In [ ]:
# 1st Attempt: Using all features

# feature_columns = [
#     'flesch_kincaid_score', 'sentence_count', 'syllables_per_word',
#     'polysyllable_count', 'flesch_reading_ease', 'automated_readability_index',
#     'dale_chall_readability_score', 'difficult_words', 'linsear_write_formula',
#     'spache_readability', 'reading_time'
# ]

# X = df[feature_columns]
# y = df['fk_estimated_uk_age_num']

## Attempt 2: Removing Flesch-Kincaid

Flesch-Kincaid score is excluded from the feature set. The model now relies on other features, leading to a performance drop but greater generalisability.

In [ ]:
# 2nd Attempt: Removing 'flesch_kincaid_score' from feature columns

# feature_columns = [
#     'sentence_count', 'syllables_per_word',
#     'polysyllable_count', 'flesch_reading_ease', 'automated_readability_index',
#     'dale_chall_readability_score', 'difficult_words', 'linsear_write_formula',
#     'spache_readability', 'reading_time'
# ]

# X = df[feature_columns]
# y = df['fk_estimated_uk_age_num']

## Attempt 3: Expanded Feature Set

In this attempt, I added additional engineered features (such as word count, average word length, percent complex/difficult words) to diversify the predictors and improve model robustness.

In [ ]:
# 3rd Attempt: Expanding Feature Set after Removing Flesch-Kincaid

feature_columns = [
    'word_count', 'average_word_length', 'percent_complex_words', 'percent_difficult_words', 'sentence_count', 'syllables_per_word',
    'polysyllable_count', 'flesch_reading_ease', 'automated_readability_index',
    'dale_chall_readability_score', 'difficult_words', 'linsear_write_formula',
    'spache_readability', 'reading_time'
]

X = df[feature_columns]
y = df['fk_estimated_uk_age_num']

## Model Training: Tuning Tree Depth

In [ ]:
# No max depth given, allowing the tree to grow fully
# regression_decision_tree = DecisionTreeRegressor(random_state=42)
# regression_decision_tree.fit(X_train, y_train)

In [ ]:
# Used max depth of 3 to prevent overfitting
# regression_decision_tree = DecisionTreeRegressor(max_depth=3, random_state=42)
# regression_decision_tree.fit(X_train, y_train)

## Visualise Tree Depth vs R²

Having tried no max_depth and max_depth of 3, I used plot training and test R² scores for different tree depths to visualize the bias-variance tradeoff and select the optimal tree depth.

In [ ]:
training = []
test = []

for i in range(1, 21):
    model = DecisionTreeRegressor(max_depth=i, random_state=42)
    model.fit(X_train, y_train)
    train_predictions = model.predict(X_train)
    test_predictions = model.predict(X_test)
    training.append(r2_score(y_train, train_predictions))
    test.append(r2_score(y_test, test_predictions))

plt.plot(range(1, 21), training, label='Train R^2')
plt.plot(range(1, 21), test, label='Test R^2')
plt.xlabel('Max Depth')
plt.ylabel('R^2 Score')
plt.legend()
plt.title('Decision Tree Depth vs R^2')
plt.show()

In [ ]:
best_depth = range(1, 21)[test.index(max(test))]
print(f'Best max_depth for test R^2: {best_depth}, Test R^2: {max(test):.3f}')

The best max_depth came in at 10

In [ ]:
# Used max depth of 10 to allow for more complexity in the model based on the expanded feature set
regression_decision_tree = DecisionTreeRegressor(max_depth=10, random_state=42)
regression_decision_tree.fit(X_train, y_train)

In [ ]:
# Evaluate the model with max_depth=10 (from your exploration)
tree_depth10 = DecisionTreeRegressor(max_depth=10, random_state=42)
tree_depth10.fit(X_train, y_train)
y_pred_10 = tree_depth10.predict(X_test)

mae_10 = mean_absolute_error(y_test, y_pred_10)
mse_10 = mean_squared_error(y_test, y_pred_10)
rmse_10 = mse_10 ** 0.5
r2_10 = r2_score(y_test, y_pred_10)

print("Results for DecisionTreeRegressor with max_depth=10:")
print(f"MAE: {mae_10:.2f}, MSE: {mse_10:.2f}, RMSE: {rmse_10:.2f}, R^2: {r2_10:.2f}")

# Evaluate the best_tree from GridSearchCV
y_pred_best = best_tree.predict(X_test)

mae_best = mean_absolute_error(y_test, y_pred_best)
mse_best = mean_squared_error(y_test, y_pred_best)
rmse_best = mse_best ** 0.5
r2_best = r2_score(y_test, y_pred_best)

print("\nResults for DecisionTreeRegressor with best max_depth (GridSearchCV):")
print(f"MAE: {mae_best:.2f}, MSE: {mse_best:.2f}, RMSE: {rmse_best:.2f}, R^2: {r2_best:.2f}")

-----------------

## Random Forest Regressor

Started with no max depth, then ran the best depth finder. This still adds an element of bias though.

In [ ]:
# Train the Random Forest model with no max_depth
# random_forest_tree = RandomForestRegressor(random_state=42)
# random_forest_tree.fit(X_train, y_train)

## Visualise Tree Depth vs R²

In [ ]:
training = []
test = []

for i in range(1, 21):
    model = RandomForestRegressor(max_depth=i, random_state=42)
    model.fit(X_train, y_train)
    train_predictions = model.predict(X_train)
    test_predictions = model.predict(X_test)
    training.append(r2_score(y_train, train_predictions))
    test.append(r2_score(y_test, test_predictions))

plt.plot(range(1, 21), training, label='Train R^2')
plt.plot(range(1, 21), test, label='Test R^2')
plt.xlabel('Max Depth')
plt.ylabel('R^2 Score')
plt.legend()
plt.title('Decision Tree Depth vs R^2')
plt.show()

In [ ]:
best_depth = range(1, 21)[test.index(max(test))]
print(f'Best max_depth for test R^2: {best_depth}, Test R^2: {max(test):.3f}')

Best depth was 11

In [ ]:
random_forest_tree = RandomForestRegressor(max_depth=11, random_state=42)
random_forest_tree.fit(X_train, y_train)